# Gemma 4 Developer Agent — Improved Agent (v2)

**Before running on Kaggle:**
1. Settings → Accelerator: **GPU L4 x4**, Internet: **off**.
2. Add inputs: the competition `gemma-4-developer-agent`, the dataset `metric/gemma-4-developer-agent-wheelhouse`, and the model `google/gemma-4` → `gemma-4-31b-it-qat-w4a16-ct` (version 2).
3. Run All. The final cell writes `/kaggle/working/submission.zip`. Submit it from the Output tab or upload it on **Submit Predictions**.

**What changed vs. the starter:**
- A detailed system prompt with a locate → reproduce → fix → test → clean up → submit workflow.
- Per-task budget of 5 minutes. About 120 hidden tasks must fit in 12 hours, so this leaves headroom.
- Placeholder LoRA adapters removed (base model only). They are restored automatically if the config still references them.
- Local evaluation on ~22 tasks spread across all repos, with a failure analysis comparing your patch to the reference fix.

Set `RUN_LOCAL_EVAL = False` in section 5 if you only want to build the zip quickly (no GPU time spent on evaluation).

## 1. Environment Configuration and Package Installation

In [ ]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

## 2. Competition Dataset and Sample Submission

In [ ]:
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# Copy sample_submission to a writable directory for customization and packaging
SAMPLE_SUBMISSION_SRC = DATA_DIR / 'sample_submission'
AGENT_DIR = WORKING_DIR / 'sample_submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(SAMPLE_SUBMISSION_SRC, AGENT_DIR)

# Ensure sampling.yaml omits thinking_level (mapped to OpenAI reasoning_effort by LiteLLM)
(AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
    'temperature: 0.2\n'
    'top_p: 0.95\n'
    'max_output_tokens: 16384\n'
    'thinking_config:\n'
    '  thinking_budget: 4096\n'
    '  include_thoughts: true\n',
    encoding='utf-8',
)

# Load tasks from the published dataset
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
for t in tasks[:5]:
    print(f'  - {t.instance_id} ({t.repo} @ {t.base_commit[:8]})')

## 2b. Agent Configuration (prompt, budgets, adapters)

In [ ]:
import re
import yaml

# =====================================================================
# Settings you may want to tweak between experiments
# =====================================================================
REMOVE_PLACEHOLDER_ADAPTERS = True   # sample adapters are tiny placeholders
MAX_TIME_MINUTES = 5                 # ~120 hidden tasks must fit in 12 h total
MAX_TOOL_CALLS = 80
MAX_TURNS = 80

# ---- 1. Show what the sample submission contains ----
print("Files in submission:")
for p in sorted(AGENT_DIR.rglob("*")):
    if p.is_file():
        print(f"   {p.relative_to(AGENT_DIR)}  ({p.stat().st_size/1e6:.3f} MB)")

for name in ["agent.yaml", "sub_agents/code_analyzer.yaml"]:
    f = AGENT_DIR / name
    if f.exists():
        print(f"\n===== ORIGINAL {name} =====\n{f.read_text(encoding='utf-8')}")

# ---- 2. Remove placeholder LoRA adapters (safely) ----
if REMOVE_PLACEHOLDER_ADAPTERS and (AGENT_DIR / "adapters").exists():
    yaml_files = [AGENT_DIR / "agent.yaml", *(AGENT_DIR / "sub_agents").glob("*.yaml")]
    originals = {y: y.read_text(encoding="utf-8") for y in yaml_files}
    adapter_names = [d.name for d in (AGENT_DIR / "adapters").iterdir() if d.is_dir()]
    for y, txt in originals.items():
        y.write_text(re.sub(r"(?m)^[ \t]*adapter:[ \t]*\S+[ \t]*\n", "", txt), encoding="utf-8")
    still_referenced = [
        n for n in adapter_names
        if any(n in y.read_text(encoding="utf-8") for y in yaml_files)
    ]
    if still_referenced:
        for y, txt in originals.items():
            y.write_text(txt, encoding="utf-8")
        print(f"\n[!] Adapters still referenced elsewhere ({still_referenced}); kept them unchanged.")
    else:
        shutil.rmtree(AGENT_DIR / "adapters")
        print(f"\nRemoved placeholder adapters: {adapter_names}")

# ---- 3. Budgets that fit inside the 12-hour total ----
cfg_path = AGENT_DIR / "eval_config.yaml"
raw = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
section = raw["evaluation"] if "evaluation" in raw else raw
print("\nOLD eval_config:", dict(section))
section["max_time_minutes"] = MAX_TIME_MINUTES
section["max_tool_calls"] = MAX_TOOL_CALLS
section["max_turns"] = MAX_TURNS
cfg_path.write_text(yaml.safe_dump(raw, sort_keys=False), encoding="utf-8")
print("NEW eval_config:", dict(section))

# ---- 4. Sampling ----
(AGENT_DIR / "configs" / "sampling.yaml").write_text(
    "temperature: 0.2\n"
    "top_p: 0.95\n"
    "max_output_tokens: 8192\n"
    "thinking_config:\n"
    "  thinking_budget: 2048\n"
    "  include_thoughts: false\n",
    encoding="utf-8",
)
print("sampling.yaml updated")

# ---- 5. Detailed system prompt ----
SYSTEM_PROMPT = """\
You are an expert Python software engineer. Fix the issue described in the task by editing
the repository in your current working directory, then call submit_patch. Hidden unit tests
will be run against your patch; the task counts as solved only if those tests pass.

## Workflow (follow in order)
1. UNDERSTAND: Read the issue and hints carefully. Note the exact expected behavior and every
   function/class/parameter/error name mentioned. Hidden tests use those exact names and
   behaviors, so for feature requests implement exactly what the issue describes.
2. LOCATE: Use run_command with `grep -rn "<identifier>" --include=*.py .` to find the code.
   Focus on the library package, not docs/ or examples. search_similar_code and
   get_code_neighbors work best with real symbol names you have already found
   (e.g. `package.module.ClassName`), not with English phrases.
3. READ: read_file the relevant functions using line ranges. Read the existing tests for that
   module (e.g. `ls tests | grep <module>`) to learn conventions and expected error types.
4. REPRODUCE: Write a tiny script in /tmp (e.g. /tmp/repro.py, NEVER inside the repository)
   that demonstrates the bug, and run it with `python /tmp/repro.py`.
5. FIX: Make the smallest correct change in the source code. Follow the existing code style.
   Fix the general case, not just the example in the issue. Keep the public API unchanged
   unless the issue asks for a change. Use edit_file; old_string must match the file exactly,
   including indentation.
6. VERIFY: Re-run /tmp/repro.py. Then run the related existing tests:
   `python -m pytest tests/<relevant_test_file>.py -x -q 2>&1 | tail -30`.
   If anything fails because of your change, fix it.
7. CLEAN UP: Run `git status` and `git diff`. Delete any stray files you created inside the
   repository. Do NOT modify or add files under tests/, docs/, or changelog/release notes.
8. SUBMIT: Call submit_patch.

## Rules
- Always call submit_patch before you run out of budget. An imperfect patch is better than
  none. Use get_status to check your budget; when about 70% is used, finish and submit.
- Do not explore aimlessly: if 3 searches find nothing, change strategy (broader grep, list
  the package directory, read the module __init__.py).
- Prefer targeted edit_file changes over rewriting whole files with write_file.
- Do not install packages; there is no internet. pytest is available.
- Keep command output short (pipe through head, tail or grep).
"""
(AGENT_DIR / "prompts" / "system.md").write_text(SYSTEM_PROMPT, encoding="utf-8")
print("prompts/system.md written")

agent_yaml_text = (AGENT_DIR / "agent.yaml").read_text(encoding="utf-8")
if "prompts/system.md" not in agent_yaml_text:
    print("\n[!] WARNING: agent.yaml does not !include prompts/system.md - the new prompt may not be used!")
print(f"\n===== FINAL agent.yaml =====\n{agent_yaml_text}")


## 3. Code Graph Functions

In [ ]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)

gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 5. Run Phase 1 Inference and Phase 2 Verification (local evaluation)

In [ ]:
import asyncio
import concurrent.futures
import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

# =====================================================================
RUN_LOCAL_EVAL = True      # set False to skip evaluation and only build the zip
EVAL_STRIDE = 6            # tasks[::6] -> ~22 tasks across all 4 repos (~1-2 h on L4x4)
# =====================================================================


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None

SAMPLE_TASKS = tasks[::EVAL_STRIDE] if RUN_LOCAL_EVAL else []
print(f'Evaluating {len(SAMPLE_TASKS)} tasks:',
      pd.Series([t.repo for t in SAMPLE_TASKS]).value_counts().to_dict() if SAMPLE_TASKS else {})
limits, gen_constraints = build_submission_limits()

eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,
)

evaluator = Evaluator(eval_config)
predictions = []
results = []   # (task, result) pairs for the analysis below

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    try:
        result = run_sync(
            evaluator.evaluate_task,
            task=task,
            task_index=idx,
            total_tasks=len(SAMPLE_TASKS),
        )
    except Exception as e:  # one crashing task should not kill the whole run
        print(f'  !! evaluation crashed: {e!r}')
        continue
    results.append((task, result))
    predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(result.agent_patch or "")}, '
        f'tool_calls={result.tool_calls}, '
        f'duration={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)


## 5b. Local Score and Failure Analysis

For each failed task, compare the reference fix with the agent's patch. Did the agent edit the wrong file? Use the wrong exception? Miss an edge case? Add a rule to the system prompt for recurring problems. Change one thing at a time, and only submit when the local resolve rate improves (you have 1 submission per day).

In [ ]:
# Local score + failure analysis: compare the agent's patch with the reference fix
if results:
    rows = [{
        'id': t.instance_id,
        'repo': t.repo,
        'resolved': bool(r.resolved),
        'has_patch': bool(r.agent_patch),
        'tool_calls': r.tool_calls,
        'seconds': round(r.duration_seconds, 1),
    } for t, r in results]
    summary = pd.DataFrame(rows)
    solved = summary['resolved'].sum()
    print(f'LOCAL RESOLVE RATE: {solved}/{len(summary)} = {solved/len(summary):.1%}')
    print(f'Tasks with a patch: {summary.has_patch.sum()}/{len(summary)}')
    print(f'Avg seconds/task: {summary.seconds.mean():.0f}  (must stay well under ~360 for the 12 h limit)\n')
    display(summary.groupby('repo')[['resolved', 'has_patch']].mean().round(2))
    display(summary)

    SHOW_CHARS = 1500
    for t, r in results:
        if r.resolved:
            continue
        print('=' * 100)
        print(f'{t.instance_id} | patch chars: {len(r.agent_patch or "")} | tool calls: {r.tool_calls}')
        print('--- PROBLEM (first 600 chars) ---')
        print((getattr(t, 'problem_statement', '') or '')[:600])
        print('--- REFERENCE FIX ---')
        print((getattr(t, 'patch', '') or '')[:SHOW_CHARS])
        print('--- AGENT PATCH ---')
        print((r.agent_patch or 'NO PATCH')[:SHOW_CHARS])
else:
    print('Local evaluation skipped.')


## 6. Package Submission Archive

In [ ]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    names = [i.filename for i in infos]
    assert 'agent.yaml' in names, 'agent.yaml must be at the root of the zip'
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'
    print('Zip contents:')
    for n in names:
        print('  ', n)

print(f'\nCreated {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')
